### Importando bibliotecas

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
from collections import Counter
import time

### Dados

In [ ]:
pd.set_option('display.max_columns', None)

In [ ]:
microdados = pd.read_parquet('/content/drive/MyDrive/PIBIC/Dados/dadosCenso2018_ColunasCINE.parquet')

In [ ]:
microdados.head()

#### Criando o dataset com as novas colunas

In [ ]:
map_cine = microdados_novos_curso[['CO_CINE_AREA_GERAL', 'CO_CURSO']].drop_duplicates('CO_CURSO')

In [ ]:
microdados_final = microdados_antigos.merge(map_cine, on='CO_CURSO', how='left')

In [ ]:
map_cine_esp = microdados_novos_curso[['CO_CINE_AREA_ESPECIFICA', 'CO_CURSO']].drop_duplicates('CO_CURSO')

In [ ]:
microdados_final = microdados_final.merge(map_cine_esp, on='CO_CURSO', how='left')

In [ ]:
map_cine_det = microdados_novos_curso[['CO_CINE_AREA_DETALHADA', 'CO_CURSO']].drop_duplicates('CO_CURSO')

In [ ]:
microdados_final = microdados_final.merge(map_cine_det, on='CO_CURSO', how='left')

In [ ]:
map_cine_rot = microdados_novos_curso[['CO_CINE_ROTULO', 'CO_CURSO']].drop_duplicates('CO_CURSO')

In [ ]:
microdados_final = microdados_final.merge(map_cine_rot, on='CO_CURSO', how='left')

In [ ]:
microdados_final.head()

In [ ]:
microdados_final.describe()

In [ ]:
# Validação
resultado = microdados_final.loc[
    microdados_final['CO_CURSO'] == 118420,
    ['CO_CINE_ROTULO', 'CO_CINE_AREA_GERAL', 'CO_CINE_AREA_ESPECIFICA', 'CO_CINE_AREA_DETALHADA', 'CO_UF', 'CO_MUNICIPIO']
]

resultado

In [ ]:
microdados_final.to_parquet('/content/drive/MyDrive/PIBIC/dadosCenso2018_ColunasCINE.parquet')

### Experimento 8 - Substituição do CO_CURSO por Códigos do CINE Brasil

#### Código baseado nos conceitos matemáticos (TED)

In [ ]:
def suc_prior_deterministic(data, sensitive_attr):
    """
    Sucesso determinístico a priori do adversário em um ataque ACU.
    """
    sensitive_values = data[sensitive_attr].unique()
    return 1 if len(sensitive_values) == 1 else 0

def suc_post_deterministic(data, qid_attrs, sensitive_attr):
    """
    Sucesso determinístico a posteriori do adversário em um ataque ACU.
    """
    groups = data.groupby(qid_attrs, dropna=False)  # Agrupa pelos QID e inclui NaN como grupo distinto
    total_records = len(data)  # Número total de registros
    success_count = 0  # Contador de sucessos

    for _, group in groups:  # Percorre cada grupo
        if len(group[sensitive_attr].unique()) == 1:  # Se o grupo tem apenas 1 valor sensível
            success_count += len(group)  # Conta todos os registros do grupo como sucesso

    return success_count / total_records  # Retorna a proporção de acertos

def suc_prior_probabilistic(data, sensitive_attr):
    """
    Sucesso probabilístico a priori do adversário em um ataque ACU.
    """
    value_counts = data[sensitive_attr].value_counts()
    total_records = len(data)
    return value_counts.max() / total_records if total_records > 0 else 0

def suc_post_probabilistic(data, qid_attrs, sensitive_attr):
    """
    Sucesso probabilístico a posteriori do adversário em um ataque ACU.
    """
    groups = data.groupby(qid_attrs, dropna=False)  # Inclui NaN como grupo distinto
    total_records = len(data)
    total_success = 0

    for _, group in groups:
        group_size = len(group)
        if group_size > 0:
            value_counts = group[sensitive_attr].value_counts(dropna=False)  # Inclui NaN ao contar
            total_success += value_counts.max()

    return total_success / total_records if total_records > 0 else 0

def degradation_privacy_deterministic(data, qid_attrs, sensitive_attr):
    """
    Degradação determinística de privacidade em um ataque ACU.
    """
    prior = suc_prior_deterministic(data, sensitive_attr)
    post = suc_post_deterministic(data, qid_attrs, sensitive_attr)
    return post - prior

def degradation_privacy_probabilistic(data, qid_attrs, sensitive_attr):
    """
    Degradação probabilística de privacidade em um ataque ACU.
    """
    prior = suc_prior_probabilistic(data, sensitive_attr)
    post = suc_post_probabilistic(data, qid_attrs, sensitive_attr)
    return post / prior if prior > 0 else 0

##### Atributo Sensível [IN_FINANCIAMENTO_ESTUDANTIL]

###### Substituição CO_CURSO por CO_CINE_GERAL

In [ ]:
qid_attrs = ['CO_CINE_AREA_GERAL']
sensitive_attr = 'IN_FINANCIAMENTO_ESTUDANTIL'
data = microdados

start_time = time.time()

sucesso_priori_deterministic = suc_prior_deterministic(data, sensitive_attr)
sucesso_priori_probabilistic = suc_prior_probabilistic(data, sensitive_attr)
sucesso_post_deterministico = suc_post_deterministic(data, qid_attrs, sensitive_attr)
sucesso_post_probabilistico = suc_post_probabilistic(data, qid_attrs, sensitive_attr)

end_time = time.time()
execution_time = end_time - start_time

print("Sucesso determinístico a priori:", sucesso_priori_deterministic)
print("Sucesso determinístico a posteriori:", sucesso_post_deterministico)
print("Degradação determinística de privacidade:", sucesso_post_deterministico - sucesso_priori_deterministic)

print("Sucesso probabilístico a priori:", sucesso_priori_probabilistic)
print("Sucesso probabilístico a posteriori:", sucesso_post_probabilistico)
print("Degradação probabilística de privacidade:", sucesso_post_probabilistico / sucesso_priori_probabilistic)

print("Tempo de execução:", execution_time, "segundos")

In [ ]:
qid_attrs = ['CO_UF', 'CO_CINE_AREA_GERAL']
sensitive_attr = 'IN_FINANCIAMENTO_ESTUDANTIL'
data = microdados

start_time = time.time()

sucesso_priori_deterministic = suc_prior_deterministic(data, sensitive_attr)
sucesso_priori_probabilistic = suc_prior_probabilistic(data, sensitive_attr)
sucesso_post_deterministico = suc_post_deterministic(data, qid_attrs, sensitive_attr)
sucesso_post_probabilistico = suc_post_probabilistic(data, qid_attrs, sensitive_attr)

end_time = time.time()
execution_time = end_time - start_time

print("Sucesso determinístico a priori:", sucesso_priori_deterministic)
print("Sucesso determinístico a posteriori:", sucesso_post_deterministico)
print("Degradação determinística de privacidade:", sucesso_post_deterministico - sucesso_priori_deterministic)

print("Sucesso probabilístico a priori:", sucesso_priori_probabilistic)
print("Sucesso probabilístico a posteriori:", sucesso_post_probabilistico)
print("Degradação probabilística de privacidade:", sucesso_post_probabilistico / sucesso_priori_probabilistic)

print("Tempo de execução:", execution_time, "segundos")

In [ ]:
qid_attrs = ['CO_UF', 'FAIXA_IBGE', 'CO_CINE_AREA_GERAL']
sensitive_attr = 'IN_FINANCIAMENTO_ESTUDANTIL'
data = microdados

start_time = time.time()

sucesso_priori_deterministic = suc_prior_deterministic(data, sensitive_attr)
sucesso_priori_probabilistic = suc_prior_probabilistic(data, sensitive_attr)
sucesso_post_deterministico = suc_post_deterministic(data, qid_attrs, sensitive_attr)
sucesso_post_probabilistico = suc_post_probabilistic(data, qid_attrs, sensitive_attr)

end_time = time.time()
execution_time = end_time - start_time

print("Sucesso determinístico a priori:", sucesso_priori_deterministic)
print("Sucesso determinístico a posteriori:", sucesso_post_deterministico)
print("Degradação determinística de privacidade:", sucesso_post_deterministico - sucesso_priori_deterministic)

print("Sucesso probabilístico a priori:", sucesso_priori_probabilistic)
print("Sucesso probabilístico a posteriori:", sucesso_post_probabilistico)
print("Degradação probabilística de privacidade:", sucesso_post_probabilistico / sucesso_priori_probabilistic)

print("Tempo de execução:", execution_time, "segundos")

In [ ]:
qid_attrs = ['CO_UF', 'FAIXA_IBGE', 'TP_COR_RACA', 'CO_CINE_AREA_GERAL']
sensitive_attr = 'IN_FINANCIAMENTO_ESTUDANTIL'
data = microdados

start_time = time.time()

sucesso_priori_deterministic = suc_prior_deterministic(data, sensitive_attr)
sucesso_priori_probabilistic = suc_prior_probabilistic(data, sensitive_attr)
sucesso_post_deterministico = suc_post_deterministic(data, qid_attrs, sensitive_attr)
sucesso_post_probabilistico = suc_post_probabilistic(data, qid_attrs, sensitive_attr)

end_time = time.time()
execution_time = end_time - start_time

print("Sucesso determinístico a priori:", sucesso_priori_deterministic)
print("Sucesso determinístico a posteriori:", sucesso_post_deterministico)
print("Degradação determinística de privacidade:", sucesso_post_deterministico - sucesso_priori_deterministic)

print("Sucesso probabilístico a priori:", sucesso_priori_probabilistic)
print("Sucesso probabilístico a posteriori:", sucesso_post_probabilistico)
print("Degradação probabilística de privacidade:", sucesso_post_probabilistico / sucesso_priori_probabilistic)

print("Tempo de execução:", execution_time, "segundos")

In [ ]:
qid_attrs = ['CO_UF', 'FAIXA_IBGE', 'TP_COR_RACA', 'TP_SEXO', 'CO_CINE_AREA_GERAL']
sensitive_attr = 'IN_FINANCIAMENTO_ESTUDANTIL'
data = microdados

start_time = time.time()

sucesso_priori_deterministic = suc_prior_deterministic(data, sensitive_attr)
sucesso_priori_probabilistic = suc_prior_probabilistic(data, sensitive_attr)
sucesso_post_deterministico = suc_post_deterministic(data, qid_attrs, sensitive_attr)
sucesso_post_probabilistico = suc_post_probabilistic(data, qid_attrs, sensitive_attr)

end_time = time.time()
execution_time = end_time - start_time

print("Sucesso determinístico a priori:", sucesso_priori_deterministic)
print("Sucesso determinístico a posteriori:", sucesso_post_deterministico)
print("Degradação determinística de privacidade:", sucesso_post_deterministico - sucesso_priori_deterministic)

print("Sucesso probabilístico a priori:", sucesso_priori_probabilistic)
print("Sucesso probabilístico a posteriori:", sucesso_post_probabilistico)
print("Degradação probabilística de privacidade:", sucesso_post_probabilistico / sucesso_priori_probabilistic)

print("Tempo de execução:", execution_time, "segundos")

In [ ]:
qid_attrs = ['CO_UF', 'FAIXA_IBGE', 'TP_COR_RACA', 'TP_SEXO', 'TP_ESCOLA_CONCLUSAO_ENS_MEDIO', 'CO_CINE_AREA_GERAL']
sensitive_attr = 'IN_FINANCIAMENTO_ESTUDANTIL'
data = microdados

start_time = time.time()

sucesso_priori_deterministic = suc_prior_deterministic(data, sensitive_attr)
sucesso_priori_probabilistic = suc_prior_probabilistic(data, sensitive_attr)
sucesso_post_deterministico = suc_post_deterministic(data, qid_attrs, sensitive_attr)
sucesso_post_probabilistico = suc_post_probabilistic(data, qid_attrs, sensitive_attr)

end_time = time.time()
execution_time = end_time - start_time

print("Sucesso determinístico a priori:", sucesso_priori_deterministic)
print("Sucesso determinístico a posteriori:", sucesso_post_deterministico)
print("Degradação determinística de privacidade:", sucesso_post_deterministico - sucesso_priori_deterministic)

print("Sucesso probabilístico a priori:", sucesso_priori_probabilistic)
print("Sucesso probabilístico a posteriori:", sucesso_post_probabilistico)
print("Degradação probabilística de privacidade:", sucesso_post_probabilistico / sucesso_priori_probabilistic)

print("Tempo de execução:", execution_time, "segundos")

In [ ]:
qid_attrs = ['CO_UF', 'FAIXA_IBGE', 'TP_COR_RACA', 'TP_SEXO', 'TP_ESCOLA_CONCLUSAO_ENS_MEDIO', 'TP_NACIONALIDADE', 'CO_CINE_AREA_GERAL']
sensitive_attr = 'IN_FINANCIAMENTO_ESTUDANTIL'
data = microdados

start_time = time.time()

sucesso_priori_deterministic = suc_prior_deterministic(data, sensitive_attr)
sucesso_priori_probabilistic = suc_prior_probabilistic(data, sensitive_attr)
sucesso_post_deterministico = suc_post_deterministic(data, qid_attrs, sensitive_attr)
sucesso_post_probabilistico = suc_post_probabilistic(data, qid_attrs, sensitive_attr)

end_time = time.time()
execution_time = end_time - start_time

print("Sucesso determinístico a priori:", sucesso_priori_deterministic)
print("Sucesso determinístico a posteriori:", sucesso_post_deterministico)
print("Degradação determinística de privacidade:", sucesso_post_deterministico - sucesso_priori_deterministic)

print("Sucesso probabilístico a priori:", sucesso_priori_probabilistic)
print("Sucesso probabilístico a posteriori:", sucesso_post_probabilistico)
print("Degradação probabilística de privacidade:", sucesso_post_probabilistico / sucesso_priori_probabilistic)

print("Tempo de execução:", execution_time, "segundos")

In [ ]:
qid_attrs = ['CO_UF', 'FAIXA_IBGE', 'TP_COR_RACA', 'TP_SEXO', 'TP_ESCOLA_CONCLUSAO_ENS_MEDIO', 'TP_NACIONALIDADE', 'CO_PAIS_ORIGEM', 'CO_CINE_AREA_GERAL']
sensitive_attr = 'IN_FINANCIAMENTO_ESTUDANTIL'
data = microdados

start_time = time.time()

sucesso_priori_deterministic = suc_prior_deterministic(data, sensitive_attr)
sucesso_priori_probabilistic = suc_prior_probabilistic(data, sensitive_attr)
sucesso_post_deterministico = suc_post_deterministic(data, qid_attrs, sensitive_attr)
sucesso_post_probabilistico = suc_post_probabilistic(data, qid_attrs, sensitive_attr)

end_time = time.time()
execution_time = end_time - start_time

print("Sucesso determinístico a priori:", sucesso_priori_deterministic)
print("Sucesso determinístico a posteriori:", sucesso_post_deterministico)
print("Degradação determinística de privacidade:", sucesso_post_deterministico - sucesso_priori_deterministic)

print("Sucesso probabilístico a priori:", sucesso_priori_probabilistic)
print("Sucesso probabilístico a posteriori:", sucesso_post_probabilistico)
print("Degradação probabilística de privacidade:", sucesso_post_probabilistico / sucesso_priori_probabilistic)

print("Tempo de execução:", execution_time, "segundos")

###### Substituição CO_CURSO por CO_CINE_ESPECIFICA

In [ ]:
qid_attrs = ['CO_CINE_AREA_ESPECIFICA']
sensitive_attr = 'IN_FINANCIAMENTO_ESTUDANTIL'
data = microdados

start_time = time.time()

sucesso_priori_deterministic = suc_prior_deterministic(data, sensitive_attr)
sucesso_priori_probabilistic = suc_prior_probabilistic(data, sensitive_attr)
sucesso_post_deterministico = suc_post_deterministic(data, qid_attrs, sensitive_attr)
sucesso_post_probabilistico = suc_post_probabilistic(data, qid_attrs, sensitive_attr)

end_time = time.time()
execution_time = end_time - start_time

print("Sucesso determinístico a priori:", sucesso_priori_deterministic)
print("Sucesso determinístico a posteriori:", sucesso_post_deterministico)
print("Degradação determinística de privacidade:", sucesso_post_deterministico - sucesso_priori_deterministic)

print("Sucesso probabilístico a priori:", sucesso_priori_probabilistic)
print("Sucesso probabilístico a posteriori:", sucesso_post_probabilistico)
print("Degradação probabilística de privacidade:", sucesso_post_probabilistico / sucesso_priori_probabilistic)

print("Tempo de execução:", execution_time, "segundos")

In [ ]:
qid_attrs = ['CO_UF', 'CO_CINE_AREA_ESPECIFICA']
sensitive_attr = 'IN_FINANCIAMENTO_ESTUDANTIL'
data = microdados

start_time = time.time()

sucesso_priori_deterministic = suc_prior_deterministic(data, sensitive_attr)
sucesso_priori_probabilistic = suc_prior_probabilistic(data, sensitive_attr)
sucesso_post_deterministico = suc_post_deterministic(data, qid_attrs, sensitive_attr)
sucesso_post_probabilistico = suc_post_probabilistic(data, qid_attrs, sensitive_attr)

end_time = time.time()
execution_time = end_time - start_time

print("Sucesso determinístico a priori:", sucesso_priori_deterministic)
print("Sucesso determinístico a posteriori:", sucesso_post_deterministico)
print("Degradação determinística de privacidade:", sucesso_post_deterministico - sucesso_priori_deterministic)

print("Sucesso probabilístico a priori:", sucesso_priori_probabilistic)
print("Sucesso probabilístico a posteriori:", sucesso_post_probabilistico)
print("Degradação probabilística de privacidade:", sucesso_post_probabilistico / sucesso_priori_probabilistic)

print("Tempo de execução:", execution_time, "segundos")

In [ ]:
qid_attrs = ['CO_UF', 'FAIXA_IBGE', 'CO_CINE_AREA_ESPECIFICA']
sensitive_attr = 'IN_FINANCIAMENTO_ESTUDANTIL'
data = microdados

start_time = time.time()

sucesso_priori_deterministic = suc_prior_deterministic(data, sensitive_attr)
sucesso_priori_probabilistic = suc_prior_probabilistic(data, sensitive_attr)
sucesso_post_deterministico = suc_post_deterministic(data, qid_attrs, sensitive_attr)
sucesso_post_probabilistico = suc_post_probabilistic(data, qid_attrs, sensitive_attr)

end_time = time.time()
execution_time = end_time - start_time

print("Sucesso determinístico a priori:", sucesso_priori_deterministic)
print("Sucesso determinístico a posteriori:", sucesso_post_deterministico)
print("Degradação determinística de privacidade:", sucesso_post_deterministico - sucesso_priori_deterministic)

print("Sucesso probabilístico a priori:", sucesso_priori_probabilistic)
print("Sucesso probabilístico a posteriori:", sucesso_post_probabilistico)
print("Degradação probabilística de privacidade:", sucesso_post_probabilistico / sucesso_priori_probabilistic)

print("Tempo de execução:", execution_time, "segundos")

In [ ]:
qid_attrs = ['CO_UF', 'FAIXA_IBGE', 'TP_COR_RACA', 'CO_CINE_AREA_ESPECIFICA']
sensitive_attr = 'IN_FINANCIAMENTO_ESTUDANTIL'
data = microdados

start_time = time.time()

sucesso_priori_deterministic = suc_prior_deterministic(data, sensitive_attr)
sucesso_priori_probabilistic = suc_prior_probabilistic(data, sensitive_attr)
sucesso_post_deterministico = suc_post_deterministic(data, qid_attrs, sensitive_attr)
sucesso_post_probabilistico = suc_post_probabilistic(data, qid_attrs, sensitive_attr)

end_time = time.time()
execution_time = end_time - start_time

print("Sucesso determinístico a priori:", sucesso_priori_deterministic)
print("Sucesso determinístico a posteriori:", sucesso_post_deterministico)
print("Degradação determinística de privacidade:", sucesso_post_deterministico - sucesso_priori_deterministic)

print("Sucesso probabilístico a priori:", sucesso_priori_probabilistic)
print("Sucesso probabilístico a posteriori:", sucesso_post_probabilistico)
print("Degradação probabilística de privacidade:", sucesso_post_probabilistico / sucesso_priori_probabilistic)

print("Tempo de execução:", execution_time, "segundos")

In [ ]:
qid_attrs = ['CO_UF', 'FAIXA_IBGE', 'TP_COR_RACA', 'TP_SEXO', 'CO_CINE_AREA_ESPECIFICA']
sensitive_attr = 'IN_FINANCIAMENTO_ESTUDANTIL'
data = microdados

start_time = time.time()

sucesso_priori_deterministic = suc_prior_deterministic(data, sensitive_attr)
sucesso_priori_probabilistic = suc_prior_probabilistic(data, sensitive_attr)
sucesso_post_deterministico = suc_post_deterministic(data, qid_attrs, sensitive_attr)
sucesso_post_probabilistico = suc_post_probabilistic(data, qid_attrs, sensitive_attr)

end_time = time.time()
execution_time = end_time - start_time

print("Sucesso determinístico a priori:", sucesso_priori_deterministic)
print("Sucesso determinístico a posteriori:", sucesso_post_deterministico)
print("Degradação determinística de privacidade:", sucesso_post_deterministico - sucesso_priori_deterministic)

print("Sucesso probabilístico a priori:", sucesso_priori_probabilistic)
print("Sucesso probabilístico a posteriori:", sucesso_post_probabilistico)
print("Degradação probabilística de privacidade:", sucesso_post_probabilistico / sucesso_priori_probabilistic)

print("Tempo de execução:", execution_time, "segundos")

In [ ]:
qid_attrs = ['CO_UF', 'FAIXA_IBGE', 'TP_COR_RACA', 'TP_SEXO', 'TP_ESCOLA_CONCLUSAO_ENS_MEDIO', 'CO_CINE_AREA_ESPECIFICA']
sensitive_attr = 'IN_FINANCIAMENTO_ESTUDANTIL'
data = microdados

start_time = time.time()

sucesso_priori_deterministic = suc_prior_deterministic(data, sensitive_attr)
sucesso_priori_probabilistic = suc_prior_probabilistic(data, sensitive_attr)
sucesso_post_deterministico = suc_post_deterministic(data, qid_attrs, sensitive_attr)
sucesso_post_probabilistico = suc_post_probabilistic(data, qid_attrs, sensitive_attr)

end_time = time.time()
execution_time = end_time - start_time

print("Sucesso determinístico a priori:", sucesso_priori_deterministic)
print("Sucesso determinístico a posteriori:", sucesso_post_deterministico)
print("Degradação determinística de privacidade:", sucesso_post_deterministico - sucesso_priori_deterministic)

print("Sucesso probabilístico a priori:", sucesso_priori_probabilistic)
print("Sucesso probabilístico a posteriori:", sucesso_post_probabilistico)
print("Degradação probabilística de privacidade:", sucesso_post_probabilistico / sucesso_priori_probabilistic)

print("Tempo de execução:", execution_time, "segundos")

In [ ]:
qid_attrs = ['CO_UF', 'FAIXA_IBGE', 'TP_COR_RACA', 'TP_SEXO', 'TP_ESCOLA_CONCLUSAO_ENS_MEDIO', 'TP_NACIONALIDADE', 'CO_CINE_AREA_ESPECIFICA']
sensitive_attr = 'IN_FINANCIAMENTO_ESTUDANTIL'
data = microdados

start_time = time.time()

sucesso_priori_deterministic = suc_prior_deterministic(data, sensitive_attr)
sucesso_priori_probabilistic = suc_prior_probabilistic(data, sensitive_attr)
sucesso_post_deterministico = suc_post_deterministic(data, qid_attrs, sensitive_attr)
sucesso_post_probabilistico = suc_post_probabilistic(data, qid_attrs, sensitive_attr)

end_time = time.time()
execution_time = end_time - start_time

print("Sucesso determinístico a priori:", sucesso_priori_deterministic)
print("Sucesso determinístico a posteriori:", sucesso_post_deterministico)
print("Degradação determinística de privacidade:", sucesso_post_deterministico - sucesso_priori_deterministic)

print("Sucesso probabilístico a priori:", sucesso_priori_probabilistic)
print("Sucesso probabilístico a posteriori:", sucesso_post_probabilistico)
print("Degradação probabilística de privacidade:", sucesso_post_probabilistico / sucesso_priori_probabilistic)

print("Tempo de execução:", execution_time, "segundos")

In [ ]:
qid_attrs = ['CO_UF', 'FAIXA_IBGE', 'TP_COR_RACA', 'TP_SEXO', 'TP_ESCOLA_CONCLUSAO_ENS_MEDIO', 'TP_NACIONALIDADE', 'CO_PAIS_ORIGEM', 'CO_CINE_AREA_ESPECIFICA']
sensitive_attr = 'IN_FINANCIAMENTO_ESTUDANTIL'
data = microdados

start_time = time.time()

sucesso_priori_deterministic = suc_prior_deterministic(data, sensitive_attr)
sucesso_priori_probabilistic = suc_prior_probabilistic(data, sensitive_attr)
sucesso_post_deterministico = suc_post_deterministic(data, qid_attrs, sensitive_attr)
sucesso_post_probabilistico = suc_post_probabilistic(data, qid_attrs, sensitive_attr)

end_time = time.time()
execution_time = end_time - start_time

print("Sucesso determinístico a priori:", sucesso_priori_deterministic)
print("Sucesso determinístico a posteriori:", sucesso_post_deterministico)
print("Degradação determinística de privacidade:", sucesso_post_deterministico - sucesso_priori_deterministic)

print("Sucesso probabilístico a priori:", sucesso_priori_probabilistic)
print("Sucesso probabilístico a posteriori:", sucesso_post_probabilistico)
print("Degradação probabilística de privacidade:", sucesso_post_probabilistico / sucesso_priori_probabilistic)

print("Tempo de execução:", execution_time, "segundos")

###### Substituição CO_CURSO por CO_CINE_DETALHADA

In [ ]:
qid_attrs = ['CO_CINE_AREA_DETALHADA']
sensitive_attr = 'IN_FINANCIAMENTO_ESTUDANTIL'
data = microdados

start_time = time.time()

sucesso_priori_deterministic = suc_prior_deterministic(data, sensitive_attr)
sucesso_priori_probabilistic = suc_prior_probabilistic(data, sensitive_attr)
sucesso_post_deterministico = suc_post_deterministic(data, qid_attrs, sensitive_attr)
sucesso_post_probabilistico = suc_post_probabilistic(data, qid_attrs, sensitive_attr)

end_time = time.time()
execution_time = end_time - start_time

print("Sucesso determinístico a priori:", sucesso_priori_deterministic)
print("Sucesso determinístico a posteriori:", sucesso_post_deterministico)
print("Degradação determinística de privacidade:", sucesso_post_deterministico - sucesso_priori_deterministic)

print("Sucesso probabilístico a priori:", sucesso_priori_probabilistic)
print("Sucesso probabilístico a posteriori:", sucesso_post_probabilistico)
print("Degradação probabilística de privacidade:", sucesso_post_probabilistico / sucesso_priori_probabilistic)

print("Tempo de execução:", execution_time, "segundos")

In [ ]:
qid_attrs = ['CO_UF', 'CO_CINE_AREA_DETALHADA']
sensitive_attr = 'IN_FINANCIAMENTO_ESTUDANTIL'
data = microdados

start_time = time.time()

sucesso_priori_deterministic = suc_prior_deterministic(data, sensitive_attr)
sucesso_priori_probabilistic = suc_prior_probabilistic(data, sensitive_attr)
sucesso_post_deterministico = suc_post_deterministic(data, qid_attrs, sensitive_attr)
sucesso_post_probabilistico = suc_post_probabilistic(data, qid_attrs, sensitive_attr)

end_time = time.time()
execution_time = end_time - start_time

print("Sucesso determinístico a priori:", sucesso_priori_deterministic)
print("Sucesso determinístico a posteriori:", sucesso_post_deterministico)
print("Degradação determinística de privacidade:", sucesso_post_deterministico - sucesso_priori_deterministic)

print("Sucesso probabilístico a priori:", sucesso_priori_probabilistic)
print("Sucesso probabilístico a posteriori:", sucesso_post_probabilistico)
print("Degradação probabilística de privacidade:", sucesso_post_probabilistico / sucesso_priori_probabilistic)

print("Tempo de execução:", execution_time, "segundos")

In [ ]:
qid_attrs = ['CO_UF', 'FAIXA_IBGE', 'CO_CINE_AREA_DETALHADA']
sensitive_attr = 'IN_FINANCIAMENTO_ESTUDANTIL'
data = microdados

start_time = time.time()

sucesso_priori_deterministic = suc_prior_deterministic(data, sensitive_attr)
sucesso_priori_probabilistic = suc_prior_probabilistic(data, sensitive_attr)
sucesso_post_deterministico = suc_post_deterministic(data, qid_attrs, sensitive_attr)
sucesso_post_probabilistico = suc_post_probabilistic(data, qid_attrs, sensitive_attr)

end_time = time.time()
execution_time = end_time - start_time

print("Sucesso determinístico a priori:", sucesso_priori_deterministic)
print("Sucesso determinístico a posteriori:", sucesso_post_deterministico)
print("Degradação determinística de privacidade:", sucesso_post_deterministico - sucesso_priori_deterministic)

print("Sucesso probabilístico a priori:", sucesso_priori_probabilistic)
print("Sucesso probabilístico a posteriori:", sucesso_post_probabilistico)
print("Degradação probabilística de privacidade:", sucesso_post_probabilistico / sucesso_priori_probabilistic)

print("Tempo de execução:", execution_time, "segundos")

In [ ]:
qid_attrs = ['CO_UF', 'FAIXA_IBGE', 'TP_COR_RACA', 'CO_CINE_AREA_DETALHADA']
sensitive_attr = 'IN_FINANCIAMENTO_ESTUDANTIL'
data = microdados

start_time = time.time()

sucesso_priori_deterministic = suc_prior_deterministic(data, sensitive_attr)
sucesso_priori_probabilistic = suc_prior_probabilistic(data, sensitive_attr)
sucesso_post_deterministico = suc_post_deterministic(data, qid_attrs, sensitive_attr)
sucesso_post_probabilistico = suc_post_probabilistic(data, qid_attrs, sensitive_attr)

end_time = time.time()
execution_time = end_time - start_time

print("Sucesso determinístico a priori:", sucesso_priori_deterministic)
print("Sucesso determinístico a posteriori:", sucesso_post_deterministico)
print("Degradação determinística de privacidade:", sucesso_post_deterministico - sucesso_priori_deterministic)

print("Sucesso probabilístico a priori:", sucesso_priori_probabilistic)
print("Sucesso probabilístico a posteriori:", sucesso_post_probabilistico)
print("Degradação probabilística de privacidade:", sucesso_post_probabilistico / sucesso_priori_probabilistic)

print("Tempo de execução:", execution_time, "segundos")

In [ ]:
qid_attrs = ['CO_UF', 'FAIXA_IBGE', 'TP_COR_RACA', 'TP_SEXO', 'CO_CINE_AREA_DETALHADA']
sensitive_attr = 'IN_FINANCIAMENTO_ESTUDANTIL'
data = microdados

start_time = time.time()

sucesso_priori_deterministic = suc_prior_deterministic(data, sensitive_attr)
sucesso_priori_probabilistic = suc_prior_probabilistic(data, sensitive_attr)
sucesso_post_deterministico = suc_post_deterministic(data, qid_attrs, sensitive_attr)
sucesso_post_probabilistico = suc_post_probabilistic(data, qid_attrs, sensitive_attr)

end_time = time.time()
execution_time = end_time - start_time

print("Sucesso determinístico a priori:", sucesso_priori_deterministic)
print("Sucesso determinístico a posteriori:", sucesso_post_deterministico)
print("Degradação determinística de privacidade:", sucesso_post_deterministico - sucesso_priori_deterministic)

print("Sucesso probabilístico a priori:", sucesso_priori_probabilistic)
print("Sucesso probabilístico a posteriori:", sucesso_post_probabilistico)
print("Degradação probabilística de privacidade:", sucesso_post_probabilistico / sucesso_priori_probabilistic)

print("Tempo de execução:", execution_time, "segundos")

In [ ]:
qid_attrs = ['CO_UF', 'FAIXA_IBGE', 'TP_COR_RACA', 'TP_SEXO', 'TP_ESCOLA_CONCLUSAO_ENS_MEDIO', 'CO_CINE_AREA_DETALHADA']
sensitive_attr = 'IN_FINANCIAMENTO_ESTUDANTIL'
data = microdados

start_time = time.time()

sucesso_priori_deterministic = suc_prior_deterministic(data, sensitive_attr)
sucesso_priori_probabilistic = suc_prior_probabilistic(data, sensitive_attr)
sucesso_post_deterministico = suc_post_deterministic(data, qid_attrs, sensitive_attr)
sucesso_post_probabilistico = suc_post_probabilistic(data, qid_attrs, sensitive_attr)

end_time = time.time()
execution_time = end_time - start_time

print("Sucesso determinístico a priori:", sucesso_priori_deterministic)
print("Sucesso determinístico a posteriori:", sucesso_post_deterministico)
print("Degradação determinística de privacidade:", sucesso_post_deterministico - sucesso_priori_deterministic)

print("Sucesso probabilístico a priori:", sucesso_priori_probabilistic)
print("Sucesso probabilístico a posteriori:", sucesso_post_probabilistico)
print("Degradação probabilística de privacidade:", sucesso_post_probabilistico / sucesso_priori_probabilistic)

print("Tempo de execução:", execution_time, "segundos")

In [ ]:
qid_attrs = ['CO_UF', 'FAIXA_IBGE', 'TP_COR_RACA', 'TP_SEXO', 'TP_ESCOLA_CONCLUSAO_ENS_MEDIO', 'TP_NACIONALIDADE', 'CO_CINE_AREA_DETALHADA']
sensitive_attr = 'IN_FINANCIAMENTO_ESTUDANTIL'
data = microdados

start_time = time.time()

sucesso_priori_deterministic = suc_prior_deterministic(data, sensitive_attr)
sucesso_priori_probabilistic = suc_prior_probabilistic(data, sensitive_attr)
sucesso_post_deterministico = suc_post_deterministic(data, qid_attrs, sensitive_attr)
sucesso_post_probabilistico = suc_post_probabilistic(data, qid_attrs, sensitive_attr)

end_time = time.time()
execution_time = end_time - start_time

print("Sucesso determinístico a priori:", sucesso_priori_deterministic)
print("Sucesso determinístico a posteriori:", sucesso_post_deterministico)
print("Degradação determinística de privacidade:", sucesso_post_deterministico - sucesso_priori_deterministic)

print("Sucesso probabilístico a priori:", sucesso_priori_probabilistic)
print("Sucesso probabilístico a posteriori:", sucesso_post_probabilistico)
print("Degradação probabilística de privacidade:", sucesso_post_probabilistico / sucesso_priori_probabilistic)

print("Tempo de execução:", execution_time, "segundos")

In [ ]:
qid_attrs = ['CO_UF', 'FAIXA_IBGE', 'TP_COR_RACA', 'TP_SEXO', 'TP_ESCOLA_CONCLUSAO_ENS_MEDIO', 'TP_NACIONALIDADE', 'CO_PAIS_ORIGEM', 'CO_CINE_AREA_DETALHADA']
sensitive_attr = 'IN_FINANCIAMENTO_ESTUDANTIL'
data = microdados

start_time = time.time()

sucesso_priori_deterministic = suc_prior_deterministic(data, sensitive_attr)
sucesso_priori_probabilistic = suc_prior_probabilistic(data, sensitive_attr)
sucesso_post_deterministico = suc_post_deterministic(data, qid_attrs, sensitive_attr)
sucesso_post_probabilistico = suc_post_probabilistic(data, qid_attrs, sensitive_attr)

end_time = time.time()
execution_time = end_time - start_time

print("Sucesso determinístico a priori:", sucesso_priori_deterministic)
print("Sucesso determinístico a posteriori:", sucesso_post_deterministico)
print("Degradação determinística de privacidade:", sucesso_post_deterministico - sucesso_priori_deterministic)

print("Sucesso probabilístico a priori:", sucesso_priori_probabilistic)
print("Sucesso probabilístico a posteriori:", sucesso_post_probabilistico)
print("Degradação probabilística de privacidade:", sucesso_post_probabilistico / sucesso_priori_probabilistic)

print("Tempo de execução:", execution_time, "segundos")

###### Substituição CO_CURSO por CO_CINE_ROTULO

In [ ]:
qid_attrs = ['CO_CINE_ROTULO']
sensitive_attr = 'IN_FINANCIAMENTO_ESTUDANTIL'
data = microdados

start_time = time.time()

sucesso_priori_deterministic = suc_prior_deterministic(data, sensitive_attr)
sucesso_priori_probabilistic = suc_prior_probabilistic(data, sensitive_attr)
sucesso_post_deterministico = suc_post_deterministic(data, qid_attrs, sensitive_attr)
sucesso_post_probabilistico = suc_post_probabilistic(data, qid_attrs, sensitive_attr)

end_time = time.time()
execution_time = end_time - start_time

print("Sucesso determinístico a priori:", sucesso_priori_deterministic)
print("Sucesso determinístico a posteriori:", sucesso_post_deterministico)
print("Degradação determinística de privacidade:", sucesso_post_deterministico - sucesso_priori_deterministic)

print("Sucesso probabilístico a priori:", sucesso_priori_probabilistic)
print("Sucesso probabilístico a posteriori:", sucesso_post_probabilistico)
print("Degradação probabilística de privacidade:", sucesso_post_probabilistico / sucesso_priori_probabilistic)

print("Tempo de execução:", execution_time, "segundos")

In [ ]:
qid_attrs = ['CO_UF', 'CO_CINE_ROTULO']
sensitive_attr = 'IN_FINANCIAMENTO_ESTUDANTIL'
data = microdados

start_time = time.time()

sucesso_priori_deterministic = suc_prior_deterministic(data, sensitive_attr)
sucesso_priori_probabilistic = suc_prior_probabilistic(data, sensitive_attr)
sucesso_post_deterministico = suc_post_deterministic(data, qid_attrs, sensitive_attr)
sucesso_post_probabilistico = suc_post_probabilistic(data, qid_attrs, sensitive_attr)

end_time = time.time()
execution_time = end_time - start_time

print("Sucesso determinístico a priori:", sucesso_priori_deterministic)
print("Sucesso determinístico a posteriori:", sucesso_post_deterministico)
print("Degradação determinística de privacidade:", sucesso_post_deterministico - sucesso_priori_deterministic)

print("Sucesso probabilístico a priori:", sucesso_priori_probabilistic)
print("Sucesso probabilístico a posteriori:", sucesso_post_probabilistico)
print("Degradação probabilística de privacidade:", sucesso_post_probabilistico / sucesso_priori_probabilistic)

print("Tempo de execução:", execution_time, "segundos")

In [ ]:
qid_attrs = ['CO_UF', 'FAIXA_IBGE', 'CO_CINE_ROTULO']
sensitive_attr = 'IN_FINANCIAMENTO_ESTUDANTIL'
data = microdados

start_time = time.time()

sucesso_priori_deterministic = suc_prior_deterministic(data, sensitive_attr)
sucesso_priori_probabilistic = suc_prior_probabilistic(data, sensitive_attr)
sucesso_post_deterministico = suc_post_deterministic(data, qid_attrs, sensitive_attr)
sucesso_post_probabilistico = suc_post_probabilistic(data, qid_attrs, sensitive_attr)

end_time = time.time()
execution_time = end_time - start_time

print("Sucesso determinístico a priori:", sucesso_priori_deterministic)
print("Sucesso determinístico a posteriori:", sucesso_post_deterministico)
print("Degradação determinística de privacidade:", sucesso_post_deterministico - sucesso_priori_deterministic)

print("Sucesso probabilístico a priori:", sucesso_priori_probabilistic)
print("Sucesso probabilístico a posteriori:", sucesso_post_probabilistico)
print("Degradação probabilística de privacidade:", sucesso_post_probabilistico / sucesso_priori_probabilistic)

print("Tempo de execução:", execution_time, "segundos")

In [ ]:
qid_attrs = ['CO_UF', 'FAIXA_IBGE', 'TP_COR_RACA', 'CO_CINE_ROTULO']
sensitive_attr = 'IN_FINANCIAMENTO_ESTUDANTIL'
data = microdados

start_time = time.time()

sucesso_priori_deterministic = suc_prior_deterministic(data, sensitive_attr)
sucesso_priori_probabilistic = suc_prior_probabilistic(data, sensitive_attr)
sucesso_post_deterministico = suc_post_deterministic(data, qid_attrs, sensitive_attr)
sucesso_post_probabilistico = suc_post_probabilistic(data, qid_attrs, sensitive_attr)

end_time = time.time()
execution_time = end_time - start_time

print("Sucesso determinístico a priori:", sucesso_priori_deterministic)
print("Sucesso determinístico a posteriori:", sucesso_post_deterministico)
print("Degradação determinística de privacidade:", sucesso_post_deterministico - sucesso_priori_deterministic)

print("Sucesso probabilístico a priori:", sucesso_priori_probabilistic)
print("Sucesso probabilístico a posteriori:", sucesso_post_probabilistico)
print("Degradação probabilística de privacidade:", sucesso_post_probabilistico / sucesso_priori_probabilistic)

print("Tempo de execução:", execution_time, "segundos")

In [ ]:
qid_attrs = ['CO_UF', 'FAIXA_IBGE', 'TP_COR_RACA', 'TP_SEXO', 'CO_CINE_ROTULO']
sensitive_attr = 'IN_FINANCIAMENTO_ESTUDANTIL'
data = microdados

start_time = time.time()

sucesso_priori_deterministic = suc_prior_deterministic(data, sensitive_attr)
sucesso_priori_probabilistic = suc_prior_probabilistic(data, sensitive_attr)
sucesso_post_deterministico = suc_post_deterministic(data, qid_attrs, sensitive_attr)
sucesso_post_probabilistico = suc_post_probabilistic(data, qid_attrs, sensitive_attr)

end_time = time.time()
execution_time = end_time - start_time

print("Sucesso determinístico a priori:", sucesso_priori_deterministic)
print("Sucesso determinístico a posteriori:", sucesso_post_deterministico)
print("Degradação determinística de privacidade:", sucesso_post_deterministico - sucesso_priori_deterministic)

print("Sucesso probabilístico a priori:", sucesso_priori_probabilistic)
print("Sucesso probabilístico a posteriori:", sucesso_post_probabilistico)
print("Degradação probabilística de privacidade:", sucesso_post_probabilistico / sucesso_priori_probabilistic)

print("Tempo de execução:", execution_time, "segundos")

In [ ]:
qid_attrs = ['CO_UF', 'FAIXA_IBGE', 'TP_COR_RACA', 'TP_SEXO', 'TP_ESCOLA_CONCLUSAO_ENS_MEDIO', 'CO_CINE_ROTULO']
sensitive_attr = 'IN_FINANCIAMENTO_ESTUDANTIL'
data = microdados

start_time = time.time()

sucesso_priori_deterministic = suc_prior_deterministic(data, sensitive_attr)
sucesso_priori_probabilistic = suc_prior_probabilistic(data, sensitive_attr)
sucesso_post_deterministico = suc_post_deterministic(data, qid_attrs, sensitive_attr)
sucesso_post_probabilistico = suc_post_probabilistic(data, qid_attrs, sensitive_attr)

end_time = time.time()
execution_time = end_time - start_time

print("Sucesso determinístico a priori:", sucesso_priori_deterministic)
print("Sucesso determinístico a posteriori:", sucesso_post_deterministico)
print("Degradação determinística de privacidade:", sucesso_post_deterministico - sucesso_priori_deterministic)

print("Sucesso probabilístico a priori:", sucesso_priori_probabilistic)
print("Sucesso probabilístico a posteriori:", sucesso_post_probabilistico)
print("Degradação probabilística de privacidade:", sucesso_post_probabilistico / sucesso_priori_probabilistic)

print("Tempo de execução:", execution_time, "segundos")

In [ ]:
qid_attrs = ['CO_UF', 'FAIXA_IBGE', 'TP_COR_RACA', 'TP_SEXO', 'TP_ESCOLA_CONCLUSAO_ENS_MEDIO', 'TP_NACIONALIDADE', 'CO_CINE_ROTULO']
sensitive_attr = 'IN_FINANCIAMENTO_ESTUDANTIL'
data = microdados

start_time = time.time()

sucesso_priori_deterministic = suc_prior_deterministic(data, sensitive_attr)
sucesso_priori_probabilistic = suc_prior_probabilistic(data, sensitive_attr)
sucesso_post_deterministico = suc_post_deterministic(data, qid_attrs, sensitive_attr)
sucesso_post_probabilistico = suc_post_probabilistic(data, qid_attrs, sensitive_attr)

end_time = time.time()
execution_time = end_time - start_time

print("Sucesso determinístico a priori:", sucesso_priori_deterministic)
print("Sucesso determinístico a posteriori:", sucesso_post_deterministico)
print("Degradação determinística de privacidade:", sucesso_post_deterministico - sucesso_priori_deterministic)

print("Sucesso probabilístico a priori:", sucesso_priori_probabilistic)
print("Sucesso probabilístico a posteriori:", sucesso_post_probabilistico)
print("Degradação probabilística de privacidade:", sucesso_post_probabilistico / sucesso_priori_probabilistic)

print("Tempo de execução:", execution_time, "segundos")

In [ ]:
qid_attrs = ['CO_UF', 'FAIXA_IBGE', 'TP_COR_RACA', 'TP_SEXO', 'TP_ESCOLA_CONCLUSAO_ENS_MEDIO', 'TP_NACIONALIDADE', 'CO_PAIS_ORIGEM', 'CO_CINE_ROTULO']
sensitive_attr = 'IN_FINANCIAMENTO_ESTUDANTIL'
data = microdados

start_time = time.time()

sucesso_priori_deterministic = suc_prior_deterministic(data, sensitive_attr)
sucesso_priori_probabilistic = suc_prior_probabilistic(data, sensitive_attr)
sucesso_post_deterministico = suc_post_deterministic(data, qid_attrs, sensitive_attr)
sucesso_post_probabilistico = suc_post_probabilistic(data, qid_attrs, sensitive_attr)

end_time = time.time()
execution_time = end_time - start_time

print("Sucesso determinístico a priori:", sucesso_priori_deterministic)
print("Sucesso determinístico a posteriori:", sucesso_post_deterministico)
print("Degradação determinística de privacidade:", sucesso_post_deterministico - sucesso_priori_deterministic)

print("Sucesso probabilístico a priori:", sucesso_priori_probabilistic)
print("Sucesso probabilístico a posteriori:", sucesso_post_probabilistico)
print("Degradação probabilística de privacidade:", sucesso_post_probabilistico / sucesso_priori_probabilistic)

print("Tempo de execução:", execution_time, "segundos")

##### Atributo Sensível [IN_DEFICIENCIA]

###### Substituição CO_CURSO por CO_CINE_GERAL

In [ ]:
qid_attrs = ['CO_CINE_AREA_GERAL']
sensitive_attr = 'IN_DEFICIENCIA'
data = microdados

start_time = time.time()

sucesso_priori_deterministic = suc_prior_deterministic(data, sensitive_attr)
sucesso_priori_probabilistic = suc_prior_probabilistic(data, sensitive_attr)
sucesso_post_deterministico = suc_post_deterministic(data, qid_attrs, sensitive_attr)
sucesso_post_probabilistico = suc_post_probabilistic(data, qid_attrs, sensitive_attr)

end_time = time.time()
execution_time = end_time - start_time

print("Sucesso determinístico a priori:", sucesso_priori_deterministic)
print("Sucesso determinístico a posteriori:", sucesso_post_deterministico)
print("Degradação determinística de privacidade:", sucesso_post_deterministico - sucesso_priori_deterministic)

print("Sucesso probabilístico a priori:", sucesso_priori_probabilistic)
print("Sucesso probabilístico a posteriori:", sucesso_post_probabilistico)
print("Degradação probabilística de privacidade:", sucesso_post_probabilistico / sucesso_priori_probabilistic)

print("Tempo de execução:", execution_time, "segundos")

In [ ]:
qid_attrs = ['CO_UF', 'CO_CINE_AREA_GERAL']
sensitive_attr = 'IN_DEFICIENCIA'
data = microdados

start_time = time.time()

sucesso_priori_deterministic = suc_prior_deterministic(data, sensitive_attr)
sucesso_priori_probabilistic = suc_prior_probabilistic(data, sensitive_attr)
sucesso_post_deterministico = suc_post_deterministic(data, qid_attrs, sensitive_attr)
sucesso_post_probabilistico = suc_post_probabilistic(data, qid_attrs, sensitive_attr)

end_time = time.time()
execution_time = end_time - start_time

print("Sucesso determinístico a priori:", sucesso_priori_deterministic)
print("Sucesso determinístico a posteriori:", sucesso_post_deterministico)
print("Degradação determinística de privacidade:", sucesso_post_deterministico - sucesso_priori_deterministic)

print("Sucesso probabilístico a priori:", sucesso_priori_probabilistic)
print("Sucesso probabilístico a posteriori:", sucesso_post_probabilistico)
print("Degradação probabilística de privacidade:", sucesso_post_probabilistico / sucesso_priori_probabilistic)

print("Tempo de execução:", execution_time, "segundos")

In [ ]:
qid_attrs = ['CO_UF', 'FAIXA_IBGE', 'CO_CINE_AREA_GERAL']
sensitive_attr = 'IN_DEFICIENCIA'
data = microdados

start_time = time.time()

sucesso_priori_deterministic = suc_prior_deterministic(data, sensitive_attr)
sucesso_priori_probabilistic = suc_prior_probabilistic(data, sensitive_attr)
sucesso_post_deterministico = suc_post_deterministic(data, qid_attrs, sensitive_attr)
sucesso_post_probabilistico = suc_post_probabilistic(data, qid_attrs, sensitive_attr)

end_time = time.time()
execution_time = end_time - start_time

print("Sucesso determinístico a priori:", sucesso_priori_deterministic)
print("Sucesso determinístico a posteriori:", sucesso_post_deterministico)
print("Degradação determinística de privacidade:", sucesso_post_deterministico - sucesso_priori_deterministic)

print("Sucesso probabilístico a priori:", sucesso_priori_probabilistic)
print("Sucesso probabilístico a posteriori:", sucesso_post_probabilistico)
print("Degradação probabilística de privacidade:", sucesso_post_probabilistico / sucesso_priori_probabilistic)

print("Tempo de execução:", execution_time, "segundos")

In [ ]:
qid_attrs = ['CO_UF', 'FAIXA_IBGE', 'TP_COR_RACA', 'CO_CINE_AREA_GERAL']
sensitive_attr = 'IN_DEFICIENCIA'
data = microdados

start_time = time.time()

sucesso_priori_deterministic = suc_prior_deterministic(data, sensitive_attr)
sucesso_priori_probabilistic = suc_prior_probabilistic(data, sensitive_attr)
sucesso_post_deterministico = suc_post_deterministic(data, qid_attrs, sensitive_attr)
sucesso_post_probabilistico = suc_post_probabilistic(data, qid_attrs, sensitive_attr)

end_time = time.time()
execution_time = end_time - start_time

print("Sucesso determinístico a priori:", sucesso_priori_deterministic)
print("Sucesso determinístico a posteriori:", sucesso_post_deterministico)
print("Degradação determinística de privacidade:", sucesso_post_deterministico - sucesso_priori_deterministic)

print("Sucesso probabilístico a priori:", sucesso_priori_probabilistic)
print("Sucesso probabilístico a posteriori:", sucesso_post_probabilistico)
print("Degradação probabilística de privacidade:", sucesso_post_probabilistico / sucesso_priori_probabilistic)

print("Tempo de execução:", execution_time, "segundos")

In [ ]:
qid_attrs = ['CO_UF', 'FAIXA_IBGE', 'TP_COR_RACA', 'TP_SEXO', 'CO_CINE_AREA_GERAL']
sensitive_attr = 'IN_DEFICIENCIA'
data = microdados

start_time = time.time()

sucesso_priori_deterministic = suc_prior_deterministic(data, sensitive_attr)
sucesso_priori_probabilistic = suc_prior_probabilistic(data, sensitive_attr)
sucesso_post_deterministico = suc_post_deterministic(data, qid_attrs, sensitive_attr)
sucesso_post_probabilistico = suc_post_probabilistic(data, qid_attrs, sensitive_attr)

end_time = time.time()
execution_time = end_time - start_time

print("Sucesso determinístico a priori:", sucesso_priori_deterministic)
print("Sucesso determinístico a posteriori:", sucesso_post_deterministico)
print("Degradação determinística de privacidade:", sucesso_post_deterministico - sucesso_priori_deterministic)

print("Sucesso probabilístico a priori:", sucesso_priori_probabilistic)
print("Sucesso probabilístico a posteriori:", sucesso_post_probabilistico)
print("Degradação probabilística de privacidade:", sucesso_post_probabilistico / sucesso_priori_probabilistic)

print("Tempo de execução:", execution_time, "segundos")

In [ ]:
qid_attrs = ['CO_UF', 'FAIXA_IBGE', 'TP_COR_RACA', 'TP_SEXO', 'TP_ESCOLA_CONCLUSAO_ENS_MEDIO', 'CO_CINE_AREA_GERAL']
sensitive_attr = 'IN_DEFICIENCIA'
data = microdados

start_time = time.time()

sucesso_priori_deterministic = suc_prior_deterministic(data, sensitive_attr)
sucesso_priori_probabilistic = suc_prior_probabilistic(data, sensitive_attr)
sucesso_post_deterministico = suc_post_deterministic(data, qid_attrs, sensitive_attr)
sucesso_post_probabilistico = suc_post_probabilistic(data, qid_attrs, sensitive_attr)

end_time = time.time()
execution_time = end_time - start_time

print("Sucesso determinístico a priori:", sucesso_priori_deterministic)
print("Sucesso determinístico a posteriori:", sucesso_post_deterministico)
print("Degradação determinística de privacidade:", sucesso_post_deterministico - sucesso_priori_deterministic)

print("Sucesso probabilístico a priori:", sucesso_priori_probabilistic)
print("Sucesso probabilístico a posteriori:", sucesso_post_probabilistico)
print("Degradação probabilística de privacidade:", sucesso_post_probabilistico / sucesso_priori_probabilistic)

print("Tempo de execução:", execution_time, "segundos")

In [ ]:
qid_attrs = ['CO_UF', 'FAIXA_IBGE', 'TP_COR_RACA', 'TP_SEXO', 'TP_ESCOLA_CONCLUSAO_ENS_MEDIO', 'TP_NACIONALIDADE', 'CO_CINE_AREA_GERAL']
sensitive_attr = 'IN_DEFICIENCIA'
data = microdados

start_time = time.time()

sucesso_priori_deterministic = suc_prior_deterministic(data, sensitive_attr)
sucesso_priori_probabilistic = suc_prior_probabilistic(data, sensitive_attr)
sucesso_post_deterministico = suc_post_deterministic(data, qid_attrs, sensitive_attr)
sucesso_post_probabilistico = suc_post_probabilistic(data, qid_attrs, sensitive_attr)

end_time = time.time()
execution_time = end_time - start_time

print("Sucesso determinístico a priori:", sucesso_priori_deterministic)
print("Sucesso determinístico a posteriori:", sucesso_post_deterministico)
print("Degradação determinística de privacidade:", sucesso_post_deterministico - sucesso_priori_deterministic)

print("Sucesso probabilístico a priori:", sucesso_priori_probabilistic)
print("Sucesso probabilístico a posteriori:", sucesso_post_probabilistico)
print("Degradação probabilística de privacidade:", sucesso_post_probabilistico / sucesso_priori_probabilistic)

print("Tempo de execução:", execution_time, "segundos")

In [ ]:
qid_attrs = ['CO_UF', 'FAIXA_IBGE', 'TP_COR_RACA', 'TP_SEXO', 'TP_ESCOLA_CONCLUSAO_ENS_MEDIO', 'TP_NACIONALIDADE', 'CO_PAIS_ORIGEM', 'CO_CINE_AREA_GERAL']
sensitive_attr = 'IN_DEFICIENCIA'
data = microdados

start_time = time.time()

sucesso_priori_deterministic = suc_prior_deterministic(data, sensitive_attr)
sucesso_priori_probabilistic = suc_prior_probabilistic(data, sensitive_attr)
sucesso_post_deterministico = suc_post_deterministic(data, qid_attrs, sensitive_attr)
sucesso_post_probabilistico = suc_post_probabilistic(data, qid_attrs, sensitive_attr)

end_time = time.time()
execution_time = end_time - start_time

print("Sucesso determinístico a priori:", sucesso_priori_deterministic)
print("Sucesso determinístico a posteriori:", sucesso_post_deterministico)
print("Degradação determinística de privacidade:", sucesso_post_deterministico - sucesso_priori_deterministic)

print("Sucesso probabilístico a priori:", sucesso_priori_probabilistic)
print("Sucesso probabilístico a posteriori:", sucesso_post_probabilistico)
print("Degradação probabilística de privacidade:", sucesso_post_probabilistico / sucesso_priori_probabilistic)

print("Tempo de execução:", execution_time, "segundos")

###### Substituição CO_CURSO por CO_CINE_ESPECIFICA

In [ ]:
qid_attrs = ['CO_CINE_AREA_ESPECIFICA']
sensitive_attr = 'IN_DEFICIENCIA'
data = microdados

start_time = time.time()

sucesso_priori_deterministic = suc_prior_deterministic(data, sensitive_attr)
sucesso_priori_probabilistic = suc_prior_probabilistic(data, sensitive_attr)
sucesso_post_deterministico = suc_post_deterministic(data, qid_attrs, sensitive_attr)
sucesso_post_probabilistico = suc_post_probabilistic(data, qid_attrs, sensitive_attr)

end_time = time.time()
execution_time = end_time - start_time

print("Sucesso determinístico a priori:", sucesso_priori_deterministic)
print("Sucesso determinístico a posteriori:", sucesso_post_deterministico)
print("Degradação determinística de privacidade:", sucesso_post_deterministico - sucesso_priori_deterministic)

print("Sucesso probabilístico a priori:", sucesso_priori_probabilistic)
print("Sucesso probabilístico a posteriori:", sucesso_post_probabilistico)
print("Degradação probabilística de privacidade:", sucesso_post_probabilistico / sucesso_priori_probabilistic)

print("Tempo de execução:", execution_time, "segundos")

In [ ]:
qid_attrs = ['CO_UF', 'CO_CINE_AREA_ESPECIFICA']
sensitive_attr = 'IN_DEFICIENCIA'
data = microdados

start_time = time.time()

sucesso_priori_deterministic = suc_prior_deterministic(data, sensitive_attr)
sucesso_priori_probabilistic = suc_prior_probabilistic(data, sensitive_attr)
sucesso_post_deterministico = suc_post_deterministic(data, qid_attrs, sensitive_attr)
sucesso_post_probabilistico = suc_post_probabilistic(data, qid_attrs, sensitive_attr)

end_time = time.time()
execution_time = end_time - start_time

print("Sucesso determinístico a priori:", sucesso_priori_deterministic)
print("Sucesso determinístico a posteriori:", sucesso_post_deterministico)
print("Degradação determinística de privacidade:", sucesso_post_deterministico - sucesso_priori_deterministic)

print("Sucesso probabilístico a priori:", sucesso_priori_probabilistic)
print("Sucesso probabilístico a posteriori:", sucesso_post_probabilistico)
print("Degradação probabilística de privacidade:", sucesso_post_probabilistico / sucesso_priori_probabilistic)

print("Tempo de execução:", execution_time, "segundos")

In [ ]:
qid_attrs = ['CO_UF', 'FAIXA_IBGE', 'CO_CINE_AREA_ESPECIFICA']
sensitive_attr = 'IN_DEFICIENCIA'
data = microdados

start_time = time.time()

sucesso_priori_deterministic = suc_prior_deterministic(data, sensitive_attr)
sucesso_priori_probabilistic = suc_prior_probabilistic(data, sensitive_attr)
sucesso_post_deterministico = suc_post_deterministic(data, qid_attrs, sensitive_attr)
sucesso_post_probabilistico = suc_post_probabilistic(data, qid_attrs, sensitive_attr)

end_time = time.time()
execution_time = end_time - start_time

print("Sucesso determinístico a priori:", sucesso_priori_deterministic)
print("Sucesso determinístico a posteriori:", sucesso_post_deterministico)
print("Degradação determinística de privacidade:", sucesso_post_deterministico - sucesso_priori_deterministic)

print("Sucesso probabilístico a priori:", sucesso_priori_probabilistic)
print("Sucesso probabilístico a posteriori:", sucesso_post_probabilistico)
print("Degradação probabilística de privacidade:", sucesso_post_probabilistico / sucesso_priori_probabilistic)

print("Tempo de execução:", execution_time, "segundos")

In [ ]:
qid_attrs = ['CO_UF', 'FAIXA_IBGE', 'TP_COR_RACA', 'CO_CINE_AREA_ESPECIFICA']
sensitive_attr = 'IN_DEFICIENCIA'
data = microdados

start_time = time.time()

sucesso_priori_deterministic = suc_prior_deterministic(data, sensitive_attr)
sucesso_priori_probabilistic = suc_prior_probabilistic(data, sensitive_attr)
sucesso_post_deterministico = suc_post_deterministic(data, qid_attrs, sensitive_attr)
sucesso_post_probabilistico = suc_post_probabilistic(data, qid_attrs, sensitive_attr)

end_time = time.time()
execution_time = end_time - start_time

print("Sucesso determinístico a priori:", sucesso_priori_deterministic)
print("Sucesso determinístico a posteriori:", sucesso_post_deterministico)
print("Degradação determinística de privacidade:", sucesso_post_deterministico - sucesso_priori_deterministic)

print("Sucesso probabilístico a priori:", sucesso_priori_probabilistic)
print("Sucesso probabilístico a posteriori:", sucesso_post_probabilistico)
print("Degradação probabilística de privacidade:", sucesso_post_probabilistico / sucesso_priori_probabilistic)

print("Tempo de execução:", execution_time, "segundos")

In [ ]:
qid_attrs = ['CO_UF', 'FAIXA_IBGE', 'TP_COR_RACA', 'TP_SEXO', 'CO_CINE_AREA_ESPECIFICA']
sensitive_attr = 'IN_DEFICIENCIA'
data = microdados

start_time = time.time()

sucesso_priori_deterministic = suc_prior_deterministic(data, sensitive_attr)
sucesso_priori_probabilistic = suc_prior_probabilistic(data, sensitive_attr)
sucesso_post_deterministico = suc_post_deterministic(data, qid_attrs, sensitive_attr)
sucesso_post_probabilistico = suc_post_probabilistic(data, qid_attrs, sensitive_attr)

end_time = time.time()
execution_time = end_time - start_time

print("Sucesso determinístico a priori:", sucesso_priori_deterministic)
print("Sucesso determinístico a posteriori:", sucesso_post_deterministico)
print("Degradação determinística de privacidade:", sucesso_post_deterministico - sucesso_priori_deterministic)

print("Sucesso probabilístico a priori:", sucesso_priori_probabilistic)
print("Sucesso probabilístico a posteriori:", sucesso_post_probabilistico)
print("Degradação probabilística de privacidade:", sucesso_post_probabilistico / sucesso_priori_probabilistic)

print("Tempo de execução:", execution_time, "segundos")

In [ ]:
qid_attrs = ['CO_UF', 'FAIXA_IBGE', 'TP_COR_RACA', 'TP_SEXO', 'TP_ESCOLA_CONCLUSAO_ENS_MEDIO', 'CO_CINE_AREA_ESPECIFICA']
sensitive_attr = 'IN_DEFICIENCIA'
data = microdados

start_time = time.time()

sucesso_priori_deterministic = suc_prior_deterministic(data, sensitive_attr)
sucesso_priori_probabilistic = suc_prior_probabilistic(data, sensitive_attr)
sucesso_post_deterministico = suc_post_deterministic(data, qid_attrs, sensitive_attr)
sucesso_post_probabilistico = suc_post_probabilistic(data, qid_attrs, sensitive_attr)

end_time = time.time()
execution_time = end_time - start_time

print("Sucesso determinístico a priori:", sucesso_priori_deterministic)
print("Sucesso determinístico a posteriori:", sucesso_post_deterministico)
print("Degradação determinística de privacidade:", sucesso_post_deterministico - sucesso_priori_deterministic)

print("Sucesso probabilístico a priori:", sucesso_priori_probabilistic)
print("Sucesso probabilístico a posteriori:", sucesso_post_probabilistico)
print("Degradação probabilística de privacidade:", sucesso_post_probabilistico / sucesso_priori_probabilistic)

print("Tempo de execução:", execution_time, "segundos")

In [ ]:
qid_attrs = ['CO_UF', 'FAIXA_IBGE', 'TP_COR_RACA', 'TP_SEXO', 'TP_ESCOLA_CONCLUSAO_ENS_MEDIO', 'TP_NACIONALIDADE', 'CO_CINE_AREA_ESPECIFICA']
sensitive_attr = 'IN_DEFICIENCIA'
data = microdados

start_time = time.time()

sucesso_priori_deterministic = suc_prior_deterministic(data, sensitive_attr)
sucesso_priori_probabilistic = suc_prior_probabilistic(data, sensitive_attr)
sucesso_post_deterministico = suc_post_deterministic(data, qid_attrs, sensitive_attr)
sucesso_post_probabilistico = suc_post_probabilistic(data, qid_attrs, sensitive_attr)

end_time = time.time()
execution_time = end_time - start_time

print("Sucesso determinístico a priori:", sucesso_priori_deterministic)
print("Sucesso determinístico a posteriori:", sucesso_post_deterministico)
print("Degradação determinística de privacidade:", sucesso_post_deterministico - sucesso_priori_deterministic)

print("Sucesso probabilístico a priori:", sucesso_priori_probabilistic)
print("Sucesso probabilístico a posteriori:", sucesso_post_probabilistico)
print("Degradação probabilística de privacidade:", sucesso_post_probabilistico / sucesso_priori_probabilistic)

print("Tempo de execução:", execution_time, "segundos")

In [ ]:
qid_attrs = ['CO_UF', 'FAIXA_IBGE', 'TP_COR_RACA', 'TP_SEXO', 'TP_ESCOLA_CONCLUSAO_ENS_MEDIO', 'TP_NACIONALIDADE', 'CO_PAIS_ORIGEM', 'CO_CINE_AREA_ESPECIFICA']
sensitive_attr = 'IN_DEFICIENCIA'
data = microdados

start_time = time.time()

sucesso_priori_deterministic = suc_prior_deterministic(data, sensitive_attr)
sucesso_priori_probabilistic = suc_prior_probabilistic(data, sensitive_attr)
sucesso_post_deterministico = suc_post_deterministic(data, qid_attrs, sensitive_attr)
sucesso_post_probabilistico = suc_post_probabilistic(data, qid_attrs, sensitive_attr)

end_time = time.time()
execution_time = end_time - start_time

print("Sucesso determinístico a priori:", sucesso_priori_deterministic)
print("Sucesso determinístico a posteriori:", sucesso_post_deterministico)
print("Degradação determinística de privacidade:", sucesso_post_deterministico - sucesso_priori_deterministic)

print("Sucesso probabilístico a priori:", sucesso_priori_probabilistic)
print("Sucesso probabilístico a posteriori:", sucesso_post_probabilistico)
print("Degradação probabilística de privacidade:", sucesso_post_probabilistico / sucesso_priori_probabilistic)

print("Tempo de execução:", execution_time, "segundos")

###### Substituição CO_CURSO por CO_CINE_DETALHADA

In [ ]:
qid_attrs = ['CO_CINE_AREA_DETALHADA']
sensitive_attr = 'IN_DEFICIENCIA'
data = microdados

start_time = time.time()

sucesso_priori_deterministic = suc_prior_deterministic(data, sensitive_attr)
sucesso_priori_probabilistic = suc_prior_probabilistic(data, sensitive_attr)
sucesso_post_deterministico = suc_post_deterministic(data, qid_attrs, sensitive_attr)
sucesso_post_probabilistico = suc_post_probabilistic(data, qid_attrs, sensitive_attr)

end_time = time.time()
execution_time = end_time - start_time

print("Sucesso determinístico a priori:", sucesso_priori_deterministic)
print("Sucesso determinístico a posteriori:", sucesso_post_deterministico)
print("Degradação determinística de privacidade:", sucesso_post_deterministico - sucesso_priori_deterministic)

print("Sucesso probabilístico a priori:", sucesso_priori_probabilistic)
print("Sucesso probabilístico a posteriori:", sucesso_post_probabilistico)
print("Degradação probabilística de privacidade:", sucesso_post_probabilistico / sucesso_priori_probabilistic)

print("Tempo de execução:", execution_time, "segundos")

In [ ]:
qid_attrs = ['CO_UF', 'CO_CINE_AREA_DETALHADA']
sensitive_attr = 'IN_DEFICIENCIA'
data = microdados

start_time = time.time()

sucesso_priori_deterministic = suc_prior_deterministic(data, sensitive_attr)
sucesso_priori_probabilistic = suc_prior_probabilistic(data, sensitive_attr)
sucesso_post_deterministico = suc_post_deterministic(data, qid_attrs, sensitive_attr)
sucesso_post_probabilistico = suc_post_probabilistic(data, qid_attrs, sensitive_attr)

end_time = time.time()
execution_time = end_time - start_time

print("Sucesso determinístico a priori:", sucesso_priori_deterministic)
print("Sucesso determinístico a posteriori:", sucesso_post_deterministico)
print("Degradação determinística de privacidade:", sucesso_post_deterministico - sucesso_priori_deterministic)

print("Sucesso probabilístico a priori:", sucesso_priori_probabilistic)
print("Sucesso probabilístico a posteriori:", sucesso_post_probabilistico)
print("Degradação probabilística de privacidade:", sucesso_post_probabilistico / sucesso_priori_probabilistic)

print("Tempo de execução:", execution_time, "segundos")

In [ ]:
qid_attrs = ['CO_UF', 'FAIXA_IBGE', 'CO_CINE_AREA_DETALHADA']
sensitive_attr = 'IN_DEFICIENCIA'
data = microdados

start_time = time.time()

sucesso_priori_deterministic = suc_prior_deterministic(data, sensitive_attr)
sucesso_priori_probabilistic = suc_prior_probabilistic(data, sensitive_attr)
sucesso_post_deterministico = suc_post_deterministic(data, qid_attrs, sensitive_attr)
sucesso_post_probabilistico = suc_post_probabilistic(data, qid_attrs, sensitive_attr)

end_time = time.time()
execution_time = end_time - start_time

print("Sucesso determinístico a priori:", sucesso_priori_deterministic)
print("Sucesso determinístico a posteriori:", sucesso_post_deterministico)
print("Degradação determinística de privacidade:", sucesso_post_deterministico - sucesso_priori_deterministic)

print("Sucesso probabilístico a priori:", sucesso_priori_probabilistic)
print("Sucesso probabilístico a posteriori:", sucesso_post_probabilistico)
print("Degradação probabilística de privacidade:", sucesso_post_probabilistico / sucesso_priori_probabilistic)

print("Tempo de execução:", execution_time, "segundos")

In [ ]:
qid_attrs = ['CO_UF', 'FAIXA_IBGE', 'TP_COR_RACA', 'CO_CINE_AREA_DETALHADA']
sensitive_attr = 'IN_DEFICIENCIA'
data = microdados

start_time = time.time()

sucesso_priori_deterministic = suc_prior_deterministic(data, sensitive_attr)
sucesso_priori_probabilistic = suc_prior_probabilistic(data, sensitive_attr)
sucesso_post_deterministico = suc_post_deterministic(data, qid_attrs, sensitive_attr)
sucesso_post_probabilistico = suc_post_probabilistic(data, qid_attrs, sensitive_attr)

end_time = time.time()
execution_time = end_time - start_time

print("Sucesso determinístico a priori:", sucesso_priori_deterministic)
print("Sucesso determinístico a posteriori:", sucesso_post_deterministico)
print("Degradação determinística de privacidade:", sucesso_post_deterministico - sucesso_priori_deterministic)

print("Sucesso probabilístico a priori:", sucesso_priori_probabilistic)
print("Sucesso probabilístico a posteriori:", sucesso_post_probabilistico)
print("Degradação probabilística de privacidade:", sucesso_post_probabilistico / sucesso_priori_probabilistic)

print("Tempo de execução:", execution_time, "segundos")

In [ ]:
qid_attrs = ['CO_UF', 'FAIXA_IBGE', 'TP_COR_RACA', 'TP_SEXO', 'CO_CINE_AREA_DETALHADA']
sensitive_attr = 'IN_DEFICIENCIA'
data = microdados

start_time = time.time()

sucesso_priori_deterministic = suc_prior_deterministic(data, sensitive_attr)
sucesso_priori_probabilistic = suc_prior_probabilistic(data, sensitive_attr)
sucesso_post_deterministico = suc_post_deterministic(data, qid_attrs, sensitive_attr)
sucesso_post_probabilistico = suc_post_probabilistic(data, qid_attrs, sensitive_attr)

end_time = time.time()
execution_time = end_time - start_time

print("Sucesso determinístico a priori:", sucesso_priori_deterministic)
print("Sucesso determinístico a posteriori:", sucesso_post_deterministico)
print("Degradação determinística de privacidade:", sucesso_post_deterministico - sucesso_priori_deterministic)

print("Sucesso probabilístico a priori:", sucesso_priori_probabilistic)
print("Sucesso probabilístico a posteriori:", sucesso_post_probabilistico)
print("Degradação probabilística de privacidade:", sucesso_post_probabilistico / sucesso_priori_probabilistic)

print("Tempo de execução:", execution_time, "segundos")

In [ ]:
qid_attrs = ['CO_UF', 'FAIXA_IBGE', 'TP_COR_RACA', 'TP_SEXO', 'TP_ESCOLA_CONCLUSAO_ENS_MEDIO', 'CO_CINE_AREA_DETALHADA']
sensitive_attr = 'IN_DEFICIENCIA'
data = microdados

start_time = time.time()

sucesso_priori_deterministic = suc_prior_deterministic(data, sensitive_attr)
sucesso_priori_probabilistic = suc_prior_probabilistic(data, sensitive_attr)
sucesso_post_deterministico = suc_post_deterministic(data, qid_attrs, sensitive_attr)
sucesso_post_probabilistico = suc_post_probabilistic(data, qid_attrs, sensitive_attr)

end_time = time.time()
execution_time = end_time - start_time

print("Sucesso determinístico a priori:", sucesso_priori_deterministic)
print("Sucesso determinístico a posteriori:", sucesso_post_deterministico)
print("Degradação determinística de privacidade:", sucesso_post_deterministico - sucesso_priori_deterministic)

print("Sucesso probabilístico a priori:", sucesso_priori_probabilistic)
print("Sucesso probabilístico a posteriori:", sucesso_post_probabilistico)
print("Degradação probabilística de privacidade:", sucesso_post_probabilistico / sucesso_priori_probabilistic)

print("Tempo de execução:", execution_time, "segundos")

In [ ]:
qid_attrs = ['CO_UF', 'FAIXA_IBGE', 'TP_COR_RACA', 'TP_SEXO', 'TP_ESCOLA_CONCLUSAO_ENS_MEDIO', 'TP_NACIONALIDADE', 'CO_CINE_AREA_DETALHADA']
sensitive_attr = 'IN_DEFICIENCIA'
data = microdados

start_time = time.time()

sucesso_priori_deterministic = suc_prior_deterministic(data, sensitive_attr)
sucesso_priori_probabilistic = suc_prior_probabilistic(data, sensitive_attr)
sucesso_post_deterministico = suc_post_deterministic(data, qid_attrs, sensitive_attr)
sucesso_post_probabilistico = suc_post_probabilistic(data, qid_attrs, sensitive_attr)

end_time = time.time()
execution_time = end_time - start_time

print("Sucesso determinístico a priori:", sucesso_priori_deterministic)
print("Sucesso determinístico a posteriori:", sucesso_post_deterministico)
print("Degradação determinística de privacidade:", sucesso_post_deterministico - sucesso_priori_deterministic)

print("Sucesso probabilístico a priori:", sucesso_priori_probabilistic)
print("Sucesso probabilístico a posteriori:", sucesso_post_probabilistico)
print("Degradação probabilística de privacidade:", sucesso_post_probabilistico / sucesso_priori_probabilistic)

print("Tempo de execução:", execution_time, "segundos")

In [ ]:
qid_attrs = ['CO_UF', 'FAIXA_IBGE', 'TP_COR_RACA', 'TP_SEXO', 'TP_ESCOLA_CONCLUSAO_ENS_MEDIO', 'TP_NACIONALIDADE', 'CO_PAIS_ORIGEM', 'CO_CINE_AREA_DETALHADA']
sensitive_attr = 'IN_DEFICIENCIA'
data = microdados

start_time = time.time()

sucesso_priori_deterministic = suc_prior_deterministic(data, sensitive_attr)
sucesso_priori_probabilistic = suc_prior_probabilistic(data, sensitive_attr)
sucesso_post_deterministico = suc_post_deterministic(data, qid_attrs, sensitive_attr)
sucesso_post_probabilistico = suc_post_probabilistic(data, qid_attrs, sensitive_attr)

end_time = time.time()
execution_time = end_time - start_time

print("Sucesso determinístico a priori:", sucesso_priori_deterministic)
print("Sucesso determinístico a posteriori:", sucesso_post_deterministico)
print("Degradação determinística de privacidade:", sucesso_post_deterministico - sucesso_priori_deterministic)

print("Sucesso probabilístico a priori:", sucesso_priori_probabilistic)
print("Sucesso probabilístico a posteriori:", sucesso_post_probabilistico)
print("Degradação probabilística de privacidade:", sucesso_post_probabilistico / sucesso_priori_probabilistic)

print("Tempo de execução:", execution_time, "segundos")

###### Substituição CO_CURSO por CO_CINE_ROTULO

In [ ]:
qid_attrs = ['CO_CINE_ROTULO']
sensitive_attr = 'IN_DEFICIENCIA'
data = microdados

start_time = time.time()

sucesso_priori_deterministic = suc_prior_deterministic(data, sensitive_attr)
sucesso_priori_probabilistic = suc_prior_probabilistic(data, sensitive_attr)
sucesso_post_deterministico = suc_post_deterministic(data, qid_attrs, sensitive_attr)
sucesso_post_probabilistico = suc_post_probabilistic(data, qid_attrs, sensitive_attr)

end_time = time.time()
execution_time = end_time - start_time

print("Sucesso determinístico a priori:", sucesso_priori_deterministic)
print("Sucesso determinístico a posteriori:", sucesso_post_deterministico)
print("Degradação determinística de privacidade:", sucesso_post_deterministico - sucesso_priori_deterministic)

print("Sucesso probabilístico a priori:", sucesso_priori_probabilistic)
print("Sucesso probabilístico a posteriori:", sucesso_post_probabilistico)
print("Degradação probabilística de privacidade:", sucesso_post_probabilistico / sucesso_priori_probabilistic)

print("Tempo de execução:", execution_time, "segundos")

In [ ]:
qid_attrs = ['CO_UF', 'CO_CINE_ROTULO']
sensitive_attr = 'IN_DEFICIENCIA'
data = microdados

start_time = time.time()

sucesso_priori_deterministic = suc_prior_deterministic(data, sensitive_attr)
sucesso_priori_probabilistic = suc_prior_probabilistic(data, sensitive_attr)
sucesso_post_deterministico = suc_post_deterministic(data, qid_attrs, sensitive_attr)
sucesso_post_probabilistico = suc_post_probabilistic(data, qid_attrs, sensitive_attr)

end_time = time.time()
execution_time = end_time - start_time

print("Sucesso determinístico a priori:", sucesso_priori_deterministic)
print("Sucesso determinístico a posteriori:", sucesso_post_deterministico)
print("Degradação determinística de privacidade:", sucesso_post_deterministico - sucesso_priori_deterministic)

print("Sucesso probabilístico a priori:", sucesso_priori_probabilistic)
print("Sucesso probabilístico a posteriori:", sucesso_post_probabilistico)
print("Degradação probabilística de privacidade:", sucesso_post_probabilistico / sucesso_priori_probabilistic)

print("Tempo de execução:", execution_time, "segundos")

In [ ]:
qid_attrs = ['CO_UF', 'FAIXA_IBGE', 'CO_CINE_ROTULO']
sensitive_attr = 'IN_DEFICIENCIA'
data = microdados

start_time = time.time()

sucesso_priori_deterministic = suc_prior_deterministic(data, sensitive_attr)
sucesso_priori_probabilistic = suc_prior_probabilistic(data, sensitive_attr)
sucesso_post_deterministico = suc_post_deterministic(data, qid_attrs, sensitive_attr)
sucesso_post_probabilistico = suc_post_probabilistic(data, qid_attrs, sensitive_attr)

end_time = time.time()
execution_time = end_time - start_time

print("Sucesso determinístico a priori:", sucesso_priori_deterministic)
print("Sucesso determinístico a posteriori:", sucesso_post_deterministico)
print("Degradação determinística de privacidade:", sucesso_post_deterministico - sucesso_priori_deterministic)

print("Sucesso probabilístico a priori:", sucesso_priori_probabilistic)
print("Sucesso probabilístico a posteriori:", sucesso_post_probabilistico)
print("Degradação probabilística de privacidade:", sucesso_post_probabilistico / sucesso_priori_probabilistic)

print("Tempo de execução:", execution_time, "segundos")

In [ ]:
qid_attrs = ['CO_UF', 'FAIXA_IBGE', 'TP_COR_RACA', 'CO_CINE_ROTULO']
sensitive_attr = 'IN_DEFICIENCIA'
data = microdados

start_time = time.time()

sucesso_priori_deterministic = suc_prior_deterministic(data, sensitive_attr)
sucesso_priori_probabilistic = suc_prior_probabilistic(data, sensitive_attr)
sucesso_post_deterministico = suc_post_deterministic(data, qid_attrs, sensitive_attr)
sucesso_post_probabilistico = suc_post_probabilistic(data, qid_attrs, sensitive_attr)

end_time = time.time()
execution_time = end_time - start_time

print("Sucesso determinístico a priori:", sucesso_priori_deterministic)
print("Sucesso determinístico a posteriori:", sucesso_post_deterministico)
print("Degradação determinística de privacidade:", sucesso_post_deterministico - sucesso_priori_deterministic)

print("Sucesso probabilístico a priori:", sucesso_priori_probabilistic)
print("Sucesso probabilístico a posteriori:", sucesso_post_probabilistico)
print("Degradação probabilística de privacidade:", sucesso_post_probabilistico / sucesso_priori_probabilistic)

print("Tempo de execução:", execution_time, "segundos")

In [ ]:
qid_attrs = ['CO_UF', 'FAIXA_IBGE', 'TP_COR_RACA', 'TP_SEXO', 'CO_CINE_ROTULO']
sensitive_attr = 'IN_DEFICIENCIA'
data = microdados

start_time = time.time()

sucesso_priori_deterministic = suc_prior_deterministic(data, sensitive_attr)
sucesso_priori_probabilistic = suc_prior_probabilistic(data, sensitive_attr)
sucesso_post_deterministico = suc_post_deterministic(data, qid_attrs, sensitive_attr)
sucesso_post_probabilistico = suc_post_probabilistic(data, qid_attrs, sensitive_attr)

end_time = time.time()
execution_time = end_time - start_time

print("Sucesso determinístico a priori:", sucesso_priori_deterministic)
print("Sucesso determinístico a posteriori:", sucesso_post_deterministico)
print("Degradação determinística de privacidade:", sucesso_post_deterministico - sucesso_priori_deterministic)

print("Sucesso probabilístico a priori:", sucesso_priori_probabilistic)
print("Sucesso probabilístico a posteriori:", sucesso_post_probabilistico)
print("Degradação probabilística de privacidade:", sucesso_post_probabilistico / sucesso_priori_probabilistic)

print("Tempo de execução:", execution_time, "segundos")

In [ ]:
qid_attrs = ['CO_UF', 'FAIXA_IBGE', 'TP_COR_RACA', 'TP_SEXO', 'TP_ESCOLA_CONCLUSAO_ENS_MEDIO', 'CO_CINE_ROTULO']
sensitive_attr = 'IN_DEFICIENCIA'
data = microdados

start_time = time.time()

sucesso_priori_deterministic = suc_prior_deterministic(data, sensitive_attr)
sucesso_priori_probabilistic = suc_prior_probabilistic(data, sensitive_attr)
sucesso_post_deterministico = suc_post_deterministic(data, qid_attrs, sensitive_attr)
sucesso_post_probabilistico = suc_post_probabilistic(data, qid_attrs, sensitive_attr)

end_time = time.time()
execution_time = end_time - start_time

print("Sucesso determinístico a priori:", sucesso_priori_deterministic)
print("Sucesso determinístico a posteriori:", sucesso_post_deterministico)
print("Degradação determinística de privacidade:", sucesso_post_deterministico - sucesso_priori_deterministic)

print("Sucesso probabilístico a priori:", sucesso_priori_probabilistic)
print("Sucesso probabilístico a posteriori:", sucesso_post_probabilistico)
print("Degradação probabilística de privacidade:", sucesso_post_probabilistico / sucesso_priori_probabilistic)

print("Tempo de execução:", execution_time, "segundos")

In [ ]:
qid_attrs = ['CO_UF', 'FAIXA_IBGE', 'TP_COR_RACA', 'TP_SEXO', 'TP_ESCOLA_CONCLUSAO_ENS_MEDIO', 'TP_NACIONALIDADE', 'CO_CINE_ROTULO']
sensitive_attr = 'IN_DEFICIENCIA'
data = microdados

start_time = time.time()

sucesso_priori_deterministic = suc_prior_deterministic(data, sensitive_attr)
sucesso_priori_probabilistic = suc_prior_probabilistic(data, sensitive_attr)
sucesso_post_deterministico = suc_post_deterministic(data, qid_attrs, sensitive_attr)
sucesso_post_probabilistico = suc_post_probabilistic(data, qid_attrs, sensitive_attr)

end_time = time.time()
execution_time = end_time - start_time

print("Sucesso determinístico a priori:", sucesso_priori_deterministic)
print("Sucesso determinístico a posteriori:", sucesso_post_deterministico)
print("Degradação determinística de privacidade:", sucesso_post_deterministico - sucesso_priori_deterministic)

print("Sucesso probabilístico a priori:", sucesso_priori_probabilistic)
print("Sucesso probabilístico a posteriori:", sucesso_post_probabilistico)
print("Degradação probabilística de privacidade:", sucesso_post_probabilistico / sucesso_priori_probabilistic)

print("Tempo de execução:", execution_time, "segundos")

In [ ]:
qid_attrs = ['CO_UF', 'FAIXA_IBGE', 'TP_COR_RACA', 'TP_SEXO', 'TP_ESCOLA_CONCLUSAO_ENS_MEDIO', 'TP_NACIONALIDADE', 'CO_PAIS_ORIGEM', 'CO_CINE_ROTULO']
sensitive_attr = 'IN_DEFICIENCIA'
data = microdados

start_time = time.time()

sucesso_priori_deterministic = suc_prior_deterministic(data, sensitive_attr)
sucesso_priori_probabilistic = suc_prior_probabilistic(data, sensitive_attr)
sucesso_post_deterministico = suc_post_deterministic(data, qid_attrs, sensitive_attr)
sucesso_post_probabilistico = suc_post_probabilistic(data, qid_attrs, sensitive_attr)

end_time = time.time()
execution_time = end_time - start_time

print("Sucesso determinístico a priori:", sucesso_priori_deterministic)
print("Sucesso determinístico a posteriori:", sucesso_post_deterministico)
print("Degradação determinística de privacidade:", sucesso_post_deterministico - sucesso_priori_deterministic)

print("Sucesso probabilístico a priori:", sucesso_priori_probabilistic)
print("Sucesso probabilístico a posteriori:", sucesso_post_probabilistico)
print("Degradação probabilística de privacidade:", sucesso_post_probabilistico / sucesso_priori_probabilistic)

print("Tempo de execução:", execution_time, "segundos")